# <font color="red"> **3.3 Análisis de extremos**

Los **eventos extremos** se definen como condiciones poco frecuentes dentro de la variabilidad de una variable, pero su identificación depende del enfoque utilizado y del contexto de estudio. En la clase pasada definimos una manera de analizar extremos; esto mediante las anomalías estandarizadas. Además del `Z-score`, las extremos pueden caracterizarse mediante `percentiles altos`, `umbrales físicos` (valores críticos con impacto directo), `rangos intercuartiles`, etc. Por ello, **no existe una única forma de definir extremos, sino que en muchas ocasiones es necesario combinar distintas métricas para capturar tanto su rareza estadística como su relevancia física**.

En esta clase nos enfocaremos en identificar y caracterizar eventos extremos utilizando percentiles. 



In [ ]:
#Vamos a importar todas las paqueterías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature

-------

## <font color="blueviolet"> **Percentiles**

Los `percentiles` son los valores que dividen a un conjunto de datos ordenados en cien partes iguales. De manera que un percentil **indica el valor por debajo del cual se encuentra un porcentaje del conjunto de datos**; por ejemplo, el percentil 90 (P$_{90}$) representa el valor por debajo del cual se encuentra el 90 % de los datos, mientras que solo el 10 % restante es mayor. Los percentiles se representan mediante la letra P mayúscula y el subíndice del percentil. Se utilizan ampliamente para identificar valores extremos o inusuales, como aquellos por encima del percentil 95 o 99, ya que representan condiciones poco frecuentes dentro de la variabilidad de una variable.

### <font color="steelblue">**¿Cómo se calculan?**

Para calcular el valor de un percentil hay que seguir los siguientes pasos:

**1. Ordenar los datos de menor a mayor**
   
**2. Encontrar la posición del percentil:**

$$
P_{k}=\frac{(k)(n+1)}{100}
$$

donde $k$ es el percentil y $n$ el número de datos.

* Si el resultado de la fórmula es un número sin parte decimal, el percentil es el dato que está en la posición que nos proporciona la fórmula de arriba.
  
* Si el resultado de la fórmula es un número con parte decimal, el valor exacto del percentil se calcula mediante la siguiente fórmula:

$$
P=x_i+d\cdot (x_{i+1}-x_i)
$$

donde $x_i$ y $x_{i+1}$ son los números de las posiciones entre las cuales está el número obtenido por la primera fórmula, y $d$ es la parte decimal del número obtenido por la primera fórmula.


**3. Localiza el valor de la posición del percentil en tus datos**

¡Vamos a calcular percentiles!

#### <font color="orange"> **Ejemplo 1: Usando la fórmula**

In [ ]:
#Vamos a ocupar datos sobre el titanic (https://www.kaggle.com/datasets/yasserh/titanic-dataset)
titanic=pd.read_csv("titanic.csv")
titanic

In [ ]:
#Vamos a trabajar con las columnas de Pclass, Sex y Age
titanic_sliced=titanic[["Pclass","Sex","Age"]]
titanic_sliced.head()

In [ ]:
#Paso 1: Ordenamos nuestros datos de menor a mayor
titanic_sort=titanic_sliced.sort_values(by="Age")
titanic_sort

In [ ]:
#Paso 2: Vamos a sacar el percentil 1, 5, 10, 20, 50, 80, 90, 95 y 99

#Creamos una función que calcule los percentiles

def percentil(data,*percentiles):
    """
    Calcula percentiles de una lista de datos 

    Parámetros:
    data : datos
    percentiles : lista de percentiles (0-100)

    Retorna:
    Lista con los valores de los percentiles
    """

    #Creamos una lista que almacene los valores de los percentiles 
    percentiles_values=[]

    #Hacemos uso de un ciclo for
    for ip, p in enumerate(percentiles):

        #Fórmula para encontrar la posición de los percentiles
        perc=(p/100)*(len(data)-1)  #Para este caso, en vez de que sea n+1, ponemos n-1 porque Python empieza a contar desde el 0

        #Vamos a checar si la posición tienen decimales o no
        if type(perc)==int:  #En caso de que se cumpla, guarde el valor de dicha posición en la lista
            value= data.iloc[perc]
            percentiles_values.append(value)
            
        else: #Sino se cumple
            xi=data.iloc[int(perc)]
            xi_1=data.iloc[int(perc+1)]
            value=xi+(((perc+1)%1)*(xi_1-xi))
            percentiles_values.append(value)
            
            
    return percentiles_values
    
perct=np.array(percentil(titanic_sort["Age"],1, 5, 10, 20, 50, 80, 90, 95 , 99,))
perct

#### <font color="orange"> **Ejemplo 2: Usando NumPy**

Afortunadamente, NumPy cuenta con una método que nos ayuda a encontrar percentiles `np.percentile`.

In [ ]:
perc_numpy_age = np.percentile(titanic_sliced['Age'], [1, 5, 10, 20, 50, 80, 90, 95, 99])
perc_numpy_age

#### <font color="brown">**Bien ya tenemos algunos percentiles, ahora lo bueno es cómo representarlos**

#### <font color="orange"> **Ejemplo 1: Bar plots**

In [ ]:
# Supongamos que hacemos un histograma de frecuencias de todas las edades

#Defino mis intervalos
bins = np.arange(0, int(np.max(titanic_sliced['Age'])), 1)

#Grafico
plt.hist(titanic_sliced['Age'],bins = bins,  color = 'sienna',edgecolor="black",zorder=2)
plt.grid(alpha=0.3,color="gray",linestyle="--",zorder=1)
plt.ylabel("Frecuencia")
plt.xlabel("Intervalos de edades")
plt.title("Frecuencia de edades de pasajeros abordo del Titanic")

plt.show()

**Ahora supongamos que lo que queremos ver es cómo las frecuencias están repartidas basándonos en los valores de los percentiles**

In [ ]:

counts, bins, patches = plt.hist( titanic_sliced['Age'], bins = bins,edgecolor="black")
# Colorear las barras según la condición
for patch, left_side, right_side in zip(patches, bins[:-1], bins[1:]):
    bin_center = (left_side + right_side) / 2  # Calcular el centro del bin
    
    #Mayor e igual a 66 (>P99)
    if bin_center >= perc_numpy_age[-1]:
        patch.set_facecolor('purple')
    #Mayor a P95
    elif bin_center > perc_numpy_age[7]:
        patch.set_facecolor('#f17628')
    #Mayor a P90
    elif bin_center > perc_numpy_age[6]:
        patch.set_facecolor('#f79858')
    #Menor a 1
    elif bin_center < perc_numpy_age[0]:
        patch.set_facecolor('#020344')
    #Menor a 5
    elif bin_center < perc_numpy_age[1]:
        patch.set_facecolor('#155e8d')
    #Menor a 10
    elif bin_center < perc_numpy_age[2]:
        patch.set_facecolor('#28b8d5')
    #Que no cumplan ninguna de las condiciones anteriores
    else:
        patch.set_facecolor('#c0c4ca')
        
#plt.grid(alpha=0.3,color="gray",linestyle="--",zorder=1)
plt.ylabel("Frecuencia")
plt.xlabel("Intervalos de edades")
plt.title("Frecuencia de edades de pasajeros abordo del Titanic")

plt.axvline(x = perc_numpy_age[4], color = 'r', linestyle='--', alpha = 0.5)
plt.axvline(x = perc_numpy_age[2], color = 'black', linestyle='--', alpha = 0.5)
plt.axvline(x = perc_numpy_age[-3], color = 'darkblue', linestyle='--', alpha = 0.5)
plt.show()

Al observar la distribución con los percentiles, podemos observar que quizás no se repartan como intuitivamente esperábamos.
De esta manera podemos dividir los datos de manera objetiva.

Vamos a ver otro ejemplo

#### <font color="orange">**Ejemplo 2: DataFrames**

In [ ]:
#Cargar los datos

#vamos a volver a ocupar los datos de los precios de la gasolina
ruta='gas_prices.csv'
gasolina = pd.read_csv(ruta)

#Vamos a hacer una copia del dataframe gasolina 
gasolina_mxn = gasolina.copy() #Usamos .copy() cuando se quiere modificar datos sin alterar el DataFrame original.

#Los precios están en dólares, vamos a convertirlos en pesos méxicanos
gasolina_mxn.iloc[:, 1:] = gasolina.iloc[:, 1:].mul(18).round(2) #.mul(), puedo multiplicar el dataframe por un valor

In [ ]:
## Vamos a obtener ahora los percentiles de los precios

perc_gas = np.nanpercentile(gasolina_mxn.iloc[:, 1:], [5, 10, 50, 90, 95]) #np.nanpercentile(), ignora los valores nan
perc_gas

In [ ]:
# Ahora esta es una manera de iluminar sus dataframes. Usos tiene muchos, pero dependerá qué es lo que quieran lograr


# Aquí por facilidad haremos una función que otorgue un color dependiendo de si el valor que evaluamos está por encima, por debajo o 
# en un rango del percentil

def color_fondo(value, column, perc_gas):
    if value <= perc_gas[0]:
        color = '#216869'
    elif (value > perc_gas[0]) & (value <= perc_gas[1]):
        color = '#49a078'
    elif (value >= perc_gas[4]) & (value < 200):
        color = '#81171b'
    elif (value > perc_gas[3]) & (value <= perc_gas[4]):
        color = '#c75146'
    else:
        color = 'None' #Los deja en blanco
    return f'background-color: {color}'


# Aquí lo que hacemos es cambiar el estilo del dataframe

#Aplicando una función lambda
color_change=lambda x: x.apply(color_fondo, column = x.name, perc_gas = perc_gas)

gasolina_mxn.style.apply(color_change, axis = 0).format(precision=2)

#.to_excel("Df_estilos.xlsx")


In [ ]:
# Supongamos que queremos analizar un solo país i.e. Mexico

# Calculemos algunos percentiles
mex_q = np.percentile(gasolina_mxn['Mexico'], [20, 50, 80]) 


# Ajustar la función para que sólo adctúe sobre la columna de México

def color_fondo_mx(value, column, perc_gas):
  if column == 'Mexico':
    if value <= perc_gas[0]:
        color = '#216869'
    elif (value > perc_gas[0]) & (value <= perc_gas[1]):
        color = '#49a078'
    elif (value >= perc_gas[2]) :
        color = '#81171b'
    elif (value > perc_gas[1]) & (value <= perc_gas[2]):
        color = '#c75146'
    else:
        color = 'None'
  else:
    color = 'None'
  return f'background-color: {color}'


# Y aquí aplicamos el estilo
(gasolina_mxn.style
 .apply(lambda x: x.apply(color_fondo_mx, column = x.name, perc_gas = mex_q), axis = 0)
 .format(precision=2)
)#.to_excel("Df_estilos.xlsx")

------------

### <font color="purple"> **Ejercicio 1**

* Encuentren los percentiles 5, 15, 85 y 95 de 5 países los que ustedes quieran (el percentil debe de ser de cada país no de todos)
* Iluminen el dataframe de tal manera que:
    * Verde claro: por debajo del P5
    * Amarillo: entre el P5 y P15
    * Naranja: entre el P85 y P95
    * Rojo: arriba del P95


-- ----------------------- --

## <font color="steelblue"> Percentiles en Mapas

Consiste en calcular percentiles en cada punto espacial (grid) y representarlos geográficamente. En este caso, los percentiles se calculan en relación a un periodo base.

In [ ]:
# Abrir los datos

#Vamos a continuar trabajando con los datos de temperatura de australia
ruta = 'Temp_1990_2020.nc'
ds = xr.open_dataset(ruta)

In [ ]:
# Seleccionar longitud y latitud
lats = ds.latitude
lons = ds.longitude

In [ ]:
# Seleccionar la variable
tp = ds['t2m']
tp_C = tp-273.15 # convertir los datos a Celcius

In [ ]:
# Vamos a seleccionar el periodo base
temp_base = tp_C.where((tp_C.time.dt.year > 1990) & (tp_C.time.dt.year < 2001),drop=True)
temp_base

In [ ]:
# Directamente podemos calcular los percentiles

##También podemos obtener percentiles a partir de los quantiles
temp_qt = temp_base.quantile([0.5, 0.95], dim="time") 

P_5 = temp_qt.sel(quantile = 0.5)
P_95 = temp_qt.sel(quantile = 0.95)

In [ ]:
# Ahora realmente lo interesante de los percentiles es ver si algún mes fue anómalo

# Analicemos por ejemplo el 2009
comp_year = tp_C.where((tp_C.time.dt.year == 2009),drop=True)

# Ahora hace que hacer la comparación

result = comp_year.where(comp_year <= P_5,drop=True)


In [ ]:
# A diferencia de las anomalías ya no tenemos que hacer más cálculos
# Podemos plotear directamente

fig = plt.figure(figsize=(10,10))

ax = fig.add_subplot(2, 1, 1, projection=ccrs.PlateCarree())
ax.coastlines()
gl=ax.gridlines(draw_labels = True, dms = True, x_inline = False, y_inline = False, color = 'gray', alpha = 0.5, linestyle='--')
gl.top_labels = gl.right_labels = False
cs = ax.contourf(result.longitude, result.latitude, result[0], levels = np.arange(np.min(result[0]), np.max(result[0]) + 1, 0.5), cmap = 'GnBu',  transform=ccrs.PlateCarree())
plt.colorbar(cs, pad = 0.1, label = '[mm]')
plt.title('Temperaturas por debajo del P5 en el 2009', fontsize = 14)
plt.show()

In [ ]:
#Ahora vamos a ver las temperaturas por encima del P95
result2 = comp_year.where(comp_year >= P_95,drop=True)

In [ ]:
result2 = comp_year.where(comp_year >= P_95)

fig = plt.figure(figsize=(10,10))
ax = fig.add_subplot(2, 1, 1, projection=ccrs.PlateCarree())
ax.coastlines()
gl=ax.gridlines(draw_labels = True, dms = True, x_inline = False, y_inline = False, color = 'gray', alpha = 0.5, linestyle='--')
gl.top_labels = gl.right_labels = False
cs = ax.contourf(result2.longitude, result2.latitude, result2[0], levels = np.arange(np.min(result2[0]), np.max(result2[0]) + 1, 0.5), cmap = 'YlOrRd',  transform=ccrs.PlateCarree())
plt.colorbar(cs, pad = 0.1, label = '[mm]')
plt.title('Temperaturas por arriba del P95 en el 2009', fontsize = 14)
plt.show()

### <font color="purple"> **Ejercicio 2** 

Tomando como periodo base de 2010 a 2020, muestra en dónde durante el 2021 se sobrepasa el percentil 99. Explica tus resultados.